In [0]:
%sql


DROP TABLE IF EXISTS olympic_project_db.default.athletes_raw;

## creating new table for athletes##


In [0]:
df_athletes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/olympic_project_db/default/olympic_data/Athletes (5).csv")
)

df_athletes.createOrReplaceTempView("athletes_raw")

In [0]:
%sql

SELECT *
FROM athletes_raw
LIMIT 10;

PersonName,Country,Discipline
AALERUD Katrine,Norway,Cycling Road
ABAD Nestor,Spain,Artistic Gymnastics
ABAGNALE Giovanni,Italy,Rowing
ABALDE Alberto,Spain,Basketball
ABALDE Tamara,Spain,Basketball
ABALO Luc,France,Handball
ABAROA Cesar,Chile,Rowing
ABASS Abobakr,Sudan,Swimming
ABBASALI Hamideh,Islamic Republic of Iran,Karate
ABBASOV Islam,Azerbaijan,Wrestling


In [0]:
%sql

CREATE OR REPLACE TABLE olympic_project_db.default.athletes_clean AS

SELECT DISTINCT
    INITCAP(TRIM(PersonName)) AS PersonName,
    TRIM(Country) AS Country,
    TRIM(Discipline) AS Discipline
FROM athletes_raw
WHERE PersonName IS NOT NULL
  AND Country IS NOT NULL
  AND Discipline IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM olympic_project_db.default.athletes_clean
LIMIT 10;

PersonName,Country,Discipline
Abass Abobakr,Sudan,Swimming
Abdullah Rahmat Erwin,Indonesia,Weightlifting
Aghaeihajiagha Soraya,Islamic Republic of Iran,Badminton
Akhaimova Liliia,ROC,Artistic Gymnastics
Al Rashedi Mansour,Kuwait,Shooting
Al-obaidly Abdulaziz,Qatar,Swimming
Al-raimi Yasameen,Yemen,Shooting
Alarza Fernando,Spain,Triathlon
Aleksiiva Vladyslava,Ukraine,Artistic Swimming
Alemu Habitam,Ethiopia,Athletics


### creating new table for coches

In [0]:
df_coaches = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/olympic_project_db/default/olympic_data/Coaches.csv")
)

df_coaches.createOrReplaceTempView("coaches_raw")

In [0]:
%sql
CREATE OR REPLACE TABLE olympic_project_db.default.coaches_clean AS

SELECT DISTINCT
    INITCAP(TRIM(Name)) AS CoachName,
    TRIM(Country) AS Country,
    TRIM(Discipline) AS Discipline,
    COALESCE(TRIM(Event), 'Unknown') AS Event
FROM coaches_raw
WHERE Name IS NOT NULL;

num_affected_rows,num_inserted_rows


### creating new table for EntriesGender

In [0]:
df_entries = (
    spark.read
    .option("header","true")
    .option("inferSchema","true")
    .csv("/Volumes/olympic_project_db/default/olympic_data/EntriesGender.csv")
)

df_entries.createOrReplaceTempView("entriesgender_raw")

In [0]:
%sql
CREATE OR REPLACE TABLE olympic_project_db.default.entriesgender_clean AS

SELECT
    TRIM(Discipline) AS Discipline,
    CAST(Female AS INT) AS Female,
    CAST(Male AS INT) AS Male,
    CAST(Total AS INT) AS Total
FROM entriesgender_raw;

num_affected_rows,num_inserted_rows


## creating new table for medals

In [0]:
df_medals = (
    spark.read
    .option("header","true")
    .option("inferSchema","true")
    .csv("/Volumes/olympic_project_db/default/olympic_data/Medals.csv")
)

df_medals.createOrReplaceTempView("medals_raw")

In [0]:
%sql

CREATE OR REPLACE TABLE olympic_project_db.default.medals_clean AS

SELECT
    CAST(Rank AS INT) AS Rank,
    TRIM(TeamCountry) AS Country,
    CAST(Gold AS INT) AS Gold,
    CAST(Silver AS INT) AS Silver,
    CAST(Bronze AS INT) AS Bronze,
    CAST(Total AS INT) AS Total,
    CAST(`Rank by Total` AS INT) AS RankByTotal
FROM medals_raw;

num_affected_rows,num_inserted_rows


## creating new table for teams


In [0]:
df_teams = (
    spark.read
    .option("header","true")
    .option("inferSchema","true")
    .csv("/Volumes/olympic_project_db/default/olympic_data/Teams.csv")
)

df_teams.createOrReplaceTempView("teams_raw")

In [0]:
%sql
CREATE OR REPLACE TABLE olympic_project_db.default.teams_clean AS

SELECT DISTINCT
    TRIM(TeamName) AS TeamName,
    TRIM(Discipline) AS Discipline,
    TRIM(Country) AS Country,
    COALESCE(TRIM(Event), 'Unknown') AS Event
FROM teams_raw;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SHOW TABLES IN olympic_project_db.default;

database,tableName,isTemporary
default,athletes_clean,false
default,coaches_clean,false
default,entriesgender_clean,false
default,medals_clean,false
default,teams_clean,false
,athletes_raw,true
,coaches_raw,true
,entriesgender_raw,true
,medals_raw,true
,teams_raw,true


In [0]:
dbutils.fs.mkdirs("/Volumes/olympic_project_db/default/olympic_data/transformed")

True

In [0]:
spark.table("olympic_project_db.default.athletes_clean") \
    .write \
    .mode("overwrite") \
    .parquet("/Volumes/olympic_project_db/default/olympic_data/transformed/athletes")

In [0]:
spark.table("olympic_project_db.default.coaches_clean") \
    .write \
    .mode("overwrite") \
    .parquet("/Volumes/olympic_project_db/default/olympic_data/transformed/coaches")

In [0]:
spark.table("olympic_project_db.default.entriesgender_clean") \
    .write \
    .mode("overwrite") \
    .parquet("/Volumes/olympic_project_db/default/olympic_data/transformed/entriesgender")

In [0]:
spark.table("olympic_project_db.default.medals_clean") \
    .write \
    .mode("overwrite") \
    .parquet("/Volumes/olympic_project_db/default/olympic_data/transformed/medals")

In [0]:
spark.table("olympic_project_db.default.teams_clean") \
    .write \
    .mode("overwrite") \
    .parquet("/Volumes/olympic_project_db/default/olympic_data/transformed/teams")

In [0]:
display(dbutils.fs.ls("/Volumes/olympic_project_db/default/olympic_data/transformed"))

path,name,size,modificationTime
dbfs:/Volumes/olympic_project_db/default/olympic_data/transformed/athletes/,athletes/,0,1781933270000
dbfs:/Volumes/olympic_project_db/default/olympic_data/transformed/coaches/,coaches/,0,1781933286000
dbfs:/Volumes/olympic_project_db/default/olympic_data/transformed/entriesgender/,entriesgender/,0,1781933300000
dbfs:/Volumes/olympic_project_db/default/olympic_data/transformed/medals/,medals/,0,1781933314000
dbfs:/Volumes/olympic_project_db/default/olympic_data/transformed/teams/,teams/,0,1781933335000
